### Seq2Seq模型

In [5]:
import jieba
import torch
import torch.nn as nn
from collections import Counter

from jieba import tokenize

data = [
    ("你好，今天天气真好！", "Hello, the weather is nice today!"),
    ("你吃饭了吗？", "Have you eaten yet?"),
    ("深度学习很有趣。", "Deep learning is interesting."),
    ("我们一起学习吧。", "We are learning together."),
    ("这是一个测试案例。", "This is a test example.")
]


def tokenize_chinese(text):
    return list(jieba.cut(text))


def tokenize_english(text):
    return text.lower().split()


chinese_vocab = [tokenize_chinese(pair[0]) for pair in data]
english_vocab = [tokenize_english(pair[1]) for pair in data]

chinese_sentences = [tokenize_chinese(pair[0]) for pair in data]
english_sentences = [tokenize_english(pair[1]) for pair in data]




([['你好', '，', '今天天气', '真', '好', '！'],
  ['你', '吃饭', '了', '吗', '？'],
  ['深度', '学习', '很', '有趣', '。'],
  ['我们', '一起', '学习', '吧', '。'],
  ['这是', '一个', '测试', '案例', '。']],
 [['hello,', 'the', 'weather', 'is', 'nice', 'today!'],
  ['have', 'you', 'eaten', 'yet?'],
  ['deep', 'learning', 'is', 'interesting.'],
  ['we', 'are', 'learning', 'together.'],
  ['this', 'is', 'a', 'test', 'example.']])

In [17]:
special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>']


def build_vocab(sentences):
    counter = Counter()

    for sentence in sentences:
        for word in sentence:
            counter[word] += 1

    vocab = special_tokens.copy()
    for word, count in counter.items():
        if word not in special_tokens:
            vocab.append(word)

    word_to_idx = {word: idx for idx, word in enumerate(vocab)}
    return word_to_idx, vocab


chinese_word_to_idx, chinese_vocab = build_vocab([sentence for sentence in chinese_sentences])
english_word_to_idx, english_vocab = build_vocab([sentence for sentence in english_sentences])

print(chinese_vocab)
print(english_vocab)
print(chinese_word_to_idx)
print(english_word_to_idx)


['<PAD>', '<UNK>', '<BOS>', '<EOS>', '你好', '，', '今天天气', '真', '好', '！', '你', '吃饭', '了', '吗', '？', '深度', '学习', '很', '有趣', '。', '我们', '一起', '吧', '这是', '一个', '测试', '案例']
['<PAD>', '<UNK>', '<BOS>', '<EOS>', 'hello,', 'the', 'weather', 'is', 'nice', 'today!', 'have', 'you', 'eaten', 'yet?', 'deep', 'learning', 'interesting.', 'we', 'are', 'together.', 'this', 'a', 'test', 'example.']
{'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3, '你好': 4, '，': 5, '今天天气': 6, '真': 7, '好': 8, '！': 9, '你': 10, '吃饭': 11, '了': 12, '吗': 13, '？': 14, '深度': 15, '学习': 16, '很': 17, '有趣': 18, '。': 19, '我们': 20, '一起': 21, '吧': 22, '这是': 23, '一个': 24, '测试': 25, '案例': 26}
{'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3, 'hello,': 4, 'the': 5, 'weather': 6, 'is': 7, 'nice': 8, 'today!': 9, 'have': 10, 'you': 11, 'eaten': 12, 'yet?': 13, 'deep': 14, 'learning': 15, 'interesting.': 16, 'we': 17, 'are': 18, 'together.': 19, 'this': 20, 'a': 21, 'test': 22, 'example.': 23}


In [21]:
ch_vocab_size = len(chinese_vocab)
en_vocab_size = len(english_vocab)
hidden_size = 256
batch_size = 2
Learning_rate = 0.001


def tokenize(words, word_to_idx):
    return [word_to_idx.get(word, word_to_idx['<UNK>']) for word in words]


processed_data_ch = []
processed_data_en = []

for ch, en in zip(chinese_sentences, english_sentences):
    ch_numerical = [chinese_word_to_idx['<BOS>']] + tokenize(ch, chinese_word_to_idx) + [chinese_word_to_idx['<EOS>']]
    en_numerical = [english_word_to_idx['<BOS>']] + tokenize(en, english_word_to_idx) + [english_word_to_idx['<EOS>']]
    processed_data_ch.append(torch.LongTensor(ch_numerical))
    processed_data_en.append(torch.LongTensor(en_numerical))

print(processed_data_ch)
print(processed_data_en)


[tensor([2, 4, 5, 6, 7, 8, 9, 3]), tensor([ 2, 10, 11, 12, 13, 14,  3]), tensor([ 2, 15, 16, 17, 18, 19,  3]), tensor([ 2, 20, 21, 16, 22, 19,  3]), tensor([ 2, 23, 24, 25, 26, 19,  3])]
[tensor([2, 4, 5, 6, 7, 8, 9, 3]), tensor([ 2, 10, 11, 12, 13,  3]), tensor([ 2, 14, 15,  7, 16,  3]), tensor([ 2, 17, 18, 15, 19,  3]), tensor([ 2, 20,  7, 21, 22, 23,  3])]


In [25]:
from torch.nn.utils import rnn

processed_data_ch_pad = rnn.pad_sequence(processed_data_ch, batch_first=True,
                                         padding_value=chinese_word_to_idx['<PAD>'])
processed_data_en_pad = rnn.pad_sequence(processed_data_en, batch_first=True,
                                         padding_value=english_word_to_idx['<PAD>'])


tensor([[ 2,  4,  5,  6,  7,  8,  9,  3],
        [ 2, 10, 11, 12, 13,  3,  0,  0],
        [ 2, 14, 15,  7, 16,  3,  0,  0],
        [ 2, 17, 18, 15, 19,  3,  0,  0],
        [ 2, 20,  7, 21, 22, 23,  3,  0]])

In [28]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(processed_data_ch_pad, processed_data_en_pad)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for src, trg in dataloader:
    print(src)
    print(trg)
    break

tensor([[ 2, 20, 21, 16, 22, 19,  3,  0]])
tensor([[ 2, 17, 18, 15, 19,  3,  0,  0]])


In [31]:
from torch import optim
import torch.nn as nn


class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, trg, hidden):
        embedded = self.embedding(trg)
        output, hidden = self.gru(embedded, hidden)
        output = self.fc(output)
        return output, hidden


encoder = Encoder(ch_vocab_size, hidden_size)
decoder = Decoder(en_vocab_size, hidden_size)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=Learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=english_word_to_idx['<PAD>'])

epochs = 100

for step in range(epochs):
    for input, target in dataloader:
        _, hidden = encoder(input)
        decoder_input = target[:, :-1]
        decoder_target = target[:, 1:]

        decoder_output, _ = decoder(decoder_input, hidden)
        loss=criterion(decoder_output.view(-1,decoder_output.size(-1)),decoder_target.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'epoch:{step+1},loss:{loss.item()}')

epoch:1,loss:3.9157986640930176
epoch:2,loss:3.103163480758667
epoch:3,loss:0.0754786878824234
epoch:4,loss:0.7866127490997314
epoch:5,loss:0.025015532970428467
epoch:6,loss:0.0032188701443374157
epoch:7,loss:0.00242271414026618
epoch:8,loss:0.002540287096053362
epoch:9,loss:0.009083711542189121
epoch:10,loss:0.002482725540176034
epoch:11,loss:0.0012367238523438573
epoch:12,loss:0.001316827954724431
epoch:13,loss:0.0009163761860691011
epoch:14,loss:0.001135722384788096
epoch:15,loss:0.0013885516673326492
epoch:16,loss:0.0013234547805041075
epoch:17,loss:0.0012657303595915437
epoch:18,loss:0.0008816711488179862
epoch:19,loss:0.0011655244743451476
epoch:20,loss:0.0007934556342661381
epoch:21,loss:0.0011892718030139804
epoch:22,loss:0.0010368499206379056
epoch:23,loss:0.001080488320440054
epoch:24,loss:0.0009726705029606819
epoch:25,loss:0.000995043315924704
epoch:26,loss:0.0005981650901958346
epoch:27,loss:0.00042836947250179946
epoch:28,loss:0.0008594515966251493
epoch:29,loss:0.0004021

In [37]:
def translate(sentence, encoder, decoder):
    token = tokenize_chinese(sentence)
    numerical = [chinese_word_to_idx.get(word, chinese_word_to_idx['<UNK>']) for word in token]
    numerical = [chinese_word_to_idx['<BOS>']] + numerical + [chinese_word_to_idx['<EOS>']]
    src = torch.LongTensor(numerical).unsqueeze(0)
    _, hidden = encoder(src)
    trg_indexes = [english_word_to_idx['<BOS>']]

    for _ in range(10):

        trg_tensor = torch.LongTensor([trg_indexes[-1]]).unsqueeze(0)
        with torch.no_grad():
            output, hidden = decoder(trg_tensor, hidden)
        pred_token = output.argmax().item()
        trg_indexes.append(pred_token)
        if pred_token == english_word_to_idx['<EOS>']:
            break
    return ' '.join(english_vocab[idx] for idx in trg_indexes[1:-1])

test_sentence='你好，今天天气真好！'

print(translate(test_sentence,encoder,decoder))

hello, the weather is nice today!
